# Directed Graphs
Now our edges (*x*,*y*) **are ordered pairs**, so (*x*,*y*) is an edge **from** *x* **to** *y*, and **is not** the same as (*y*,*x*). 

Both BFS and DFS have natural analogues, with the resulting Trees expressing slightly different relationships between the starting node s and every node in its corresponding *T*. 

In general, and most crucially, for some node *s* in BFS(*s*) or DFS(*s*). a node *x* in the resulted connected component *T* still implies path from *s* to *x*, but not necessarily a path from *x* **to** *s*.

In [147]:
from dataclasses import dataclass
from typing import TypedDict
from __future__ import annotations

e = [(1,2),(1,3),(2,4),(2,5),(2,6),(3,5),(3,6),(3,7),(4,8),(4,9),(5,10),(5,11),
(6,11),(6,12),(7,13),(7,14),(2,1),(4,2),(3,1),(7,3),(5,2),(6,3),(8,1),(11,1),(15,1)]
n = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]


class LinkedList:
    class Node:
        def __init__(self, data):
            self.data: int = data
            self.next: Node= None

    def __init__(self):
        self.head: Node = None  # The entry point of the list
        self.tail: Node = None
    # Add a node at the end of the list
    def append(self, data: int):
        new_node = self.Node(data)
        if not self.head:
            self.head = new_node
            self.tail = new_node
            return
        current = self.head
        while current.next:  # Traverse to the last node
            current = current.next
        current.next = new_node
        self.tail = new_node
    
    def __contains__(self, data: int) -> bool:
        """Return true if there is a Node with data.
        
        Returns:
            bool: True if data is in this list, False otherwise"""
        if self.head == None:
            return False
        current: Node = self.head
        while current and current.data is not data:
            current = current.next
        if current and current.data == data:
            return True
        return False

    def __len__(self) -> int:
        if self.head == None:
            return 0;
        len = 0
        current: Node = self.head
        while current:
            current = current.next
            len+=1
        return len

@dataclass
class Graph:
    class Edges(TypedDict):
        incident: LinkedList
        outgoing: LinkedList

    edges: list[tuple[int, int]]
    nodes: list[int]
    adj_list: dict[int, Edges] = None
    V: int = 0
    E: int = 0

    def __post_init__(self):
        V = len(self.nodes)
        E = len(self.edges)
        self._adj_list()

    def _adj_list(self) -> None:
        self.adj_list = dict[int, self.Edges](
            (node, self.Edges(incident=LinkedList(), outgoing=LinkedList())) for node in self.nodes)
        for edge in self.edges:
            if(edge[0] not in self.adj_list[edge[1]]['incident']):
                self.adj_list[edge[1]]['incident'].append(edge[0])
            if(edge[1] not in self.adj_list[edge[0]]['outgoing']):
                self.adj_list[edge[0]]['outgoing'].append(edge[1])



my_tree = Graph(edges=e, nodes=n)

In [148]:
from collections import deque

def bfs(G: Graph, starting_node: int) -> list[int]:
    """Given a graph with an already defined adjacency list and a starting node in the list
    returns a list of nodes in discovered order.
    
    Returns:
        - List of int representing nodes"""
    
    discovered = list()
    q = deque[int]()
    v = set[int]()
    q.append(starting_node)
    v.add(q[0])
    # print(adj_list[5].head)
    while len(q) > 0:
        c_node = q[0]
        list_head = G.adj_list[c_node]['outgoing'].head
        while list_head is not None:
            if (list_head.data not in v):
                v.add(list_head.data)
                q.append(list_head.data)
            list_head = list_head.next
        discovered.append(c_node)
        q.popleft()
    return discovered

assert(bfs(my_tree, 1)==[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])
assert(bfs(my_tree, 2)==[2, 4, 5, 6, 1, 8, 9, 10, 11, 12, 3, 7, 13, 14])
assert(bfs(my_tree, 15)==[15, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])

In [149]:
def dfs(G: Graph, starting_node: int) -> list[int]:
    """Given a graph with an already defined adjacency list and a starting node in the list
    returns a list of nodes in discovered order.
    
    Returns:
        - List of int representing nodes"""
    
    found = list()
    q = deque[int]()
    explored = set[int]()
    parent = dict[int,int]()

    q.append(starting_node)
    parent[q[0]] = None
    while len(q) > 0:
        c_node = q[-1]
        if c_node not in explored:
            explored.add(c_node)
            if(parent[c_node]):
                found.insert(0, c_node)
            list_head = G.adj_list[c_node]['outgoing'].head
            while list_head is not None:
                q.append(list_head.data)
                parent[list_head.data] = c_node
                list_head = list_head.next
        else:
            q.pop()
    found.append(starting_node)
    return found

assert(dfs(my_tree, 1)==[10, 8, 9, 4, 2, 5, 11, 12, 6, 13, 14, 7, 3, 1])
assert(dfs(my_tree, 15)==[10, 8, 9, 4, 2, 5, 11, 12, 6, 13, 14, 7, 3, 1, 15])
assert(dfs(my_tree, 6)==[12, 13, 14, 7, 8, 9, 4, 10, 11, 5, 2, 1, 3, 6])

## Strong connectivity
"...a directed graph is strongly connected if, for every two nodes u and
v, there is a path from u to v and a path from v to u." (pg. 98)

"(3.16) If u and v are mutually reachable, and v and w are mutually reachable,
then u and w are mutually reachable." (pg. 98) which is pretty much a transitive property of connectivity.

A linear time algorithm for finding out if a directed graph is strongly connected involves running bfs on s for G and G<sup>rev</sup>. if there are nodes that are in one of the resulting set (or tree) but not the other, then the graph is not strongly connected. (pg. 98) Even if the graph is not strongly connected, the matching nodes of bfs(s) on G and G<sup>rev</sup> describe a strong component, or the strong component containing s. (pg. 99)

"(3.17) For any two nodes s and t in a directed graph, their strong components
are either identical or disjoint." (pg. 98)

In [150]:
def strongComponent(G: Graph, s: int) -> list[int]:
    connected_g = bfs(G, s)

    G_rev_edges = list[tuple[int,int]]()
    for i in range(len(G.edges)):
        G_rev_edges.append((G.edges[i][1], G.edges[i][0]))
    G_rev = Graph(G_rev_edges, G.nodes)

    connected_g_rev = bfs(G_rev, s)

    strong = [node for node in connected_g_rev if node in connected_g]
    return strong
    
assert(strongComponent(my_tree, 1)==[1, 2, 3, 8, 11, 4, 5, 7, 6])
assert(set(strongComponent(my_tree, 1))==set(strongComponent(my_tree, 2))==set(strongComponent(my_tree, 4))==set(strongComponent(my_tree, 7)))
assert(set(strongComponent(my_tree,2))!=set(strongComponent(my_tree,15)))


## 3.6 Directed Acyclic Graphs and Topological Ordering
"If an undirected graph has no cycles, then it has an extremely simple structure: each of its connected components is a tree." (pg. 99)

"If a directed graph has no cycles, we call it—naturally enough—a directed acyclic graph, or a DAG for short." (pg. 100)

"DAGs are a very common structure in computer science, because many kinds of dependency networks (...) are acyclic. <br> Thus DAGs can be used to encode precedence relations or dependencies in a natural way." (pg. 100)

"...for a directed graph G, we say that a topological ordering of G is an ordering of its nodes as v<sub>1</sub>, v<sub>2</sub>, . . . , v<sub>n</sub> so that for every edge
(v<sub>i</sub>, v<sub>j</sub>), we have i < j." (pg. 101)

"(3.18) If G has a topological ordering, then G is a DAG." (pg. 101)

"(3.19) In every DAG G, there is a node v with no incoming edges." (pg. 102)

"(3.20) If G is a DAG, then G has a topological ordering." (pg. 103)


In [151]:
import copy

def topoSort(G: Graph) -> list[int]:
    # Find node v without incident edges
    v = -1
    for node in G.nodes:
        if len(G.adj_list[node]['incident']) == 0:
            v = node
            break
    if v == -1:
        return []
    delete = True

    # delete all edges containing v
    while delete:
        i = 0
        while i < len(G.edges) and G.edges[i][0] != v and G.edges[i][1] != v:
            i+=1
        if i < len(G.edges):
            edge: tuple[int,int] = G.edges.pop(i)
            # Also, remove from other node u's adjacency list
            key = 'incident' if edge[0]==v else 'outgoing' # if deleting v==1 and edge==(1,2), we'll get the entry in 'incident' from...
            tuple_index = 1 if edge[0]==v else 0 # edge[1]==2==u. When v==1, edge==(2,1), we'll get the entry in 'outgoing' from edge[0]==2==u
            # if the node was from v to u
            prev: Node = None # prev to fix list if necessary
            head: Node = G.adj_list[edge[tuple_index]][key].head
            while head and head.data!=v:
                prev = head
                head = head.next
            # if we stopped because we found the node to remove
            if head and head.data==v:
                # if this is the head of the list
                if head == G.adj_list[edge[tuple_index]][key].head:
                    G.adj_list[edge[tuple_index]][key].head = head.next # ok if head.next is None
                    # del head
                else: # this was not the head of the list
                    prev.next = head.next # again, ok if head.next is None
            else: # We should only get here if we did not find the node to remove -> head must be None
                assert(head==None)
                # nothing to remove
        else:
            delete = False

    # remove v from adjacency list
    G.adj_list.pop(v)

    # delete v from nodes
    G.nodes.remove(v)

    return [v] + topoSort(G)


print(topoSort(Graph(edges=copy.deepcopy(my_tree.edges),nodes=copy.deepcopy(my_tree.nodes))))

my_dag = Graph(edges=[(1,4),(1,5),(1,7),(2,3),(2,5),(2,6),(3,4),(3,5),(4,5),(5,6),(5,7),(6,7)],
               nodes=[1,2,3,4,5,6,7])
assert(topoSort(my_dag)==[1,2,3,4,5,6,7])
my_dag = Graph(edges=[(5,2),(5,1),(5,7),(4,3),(4,1),(4,6),(3,2),(3,1),(2,1),(1,6),(1,7),(6,7)],
               nodes=[1,2,3,4,5,6,7])
assert(topoSort(my_dag)==[4,3,5,2,1,6,7])


[15]


In [152]:
def topoSort2(G: Graph) -> list[int]:
    # initialize 'degree' of each node and set of nodes with no incident edges S
    degree = dict[int,int]((node, 0) for node in G.nodes)
    S = set()
    for node in G.nodes:
        deg = len(G.adj_list[node]['incident'])
        if deg > 0:
            degree[node] = deg
        else:
            S.add(node)
    sorted: list[int] = []
    while len(S) > 0:
        to_delete = S.pop()
        head = G.adj_list[to_delete]['outgoing'].head
        while head:
            degree[head.data]-=1
            if degree[head.data] == 0:
                S.add(head.data)
            head = head.next
        sorted.append(to_delete)
    return sorted

my_dag = Graph(edges=[(1,4),(1,5),(1,7),(2,3),(2,5),(2,6),(3,4),(3,5),(4,5),(5,6),(5,7),(6,7)],
               nodes=[1,2,3,4,5,6,7])
print(topoSort2(my_dag))
print(my_dag.edges)
my_dag = Graph(edges=[(5,2),(5,1),(5,7),(4,3),(4,1),(4,6),(3,2),(3,1),(2,1),(1,6),(1,7),(6,7)],
               nodes=[1,2,3,4,5,6,7])
print(topoSort2(my_dag))
print(my_dag.edges)


[1, 2, 3, 4, 5, 6, 7]
[(1, 4), (1, 5), (1, 7), (2, 3), (2, 5), (2, 6), (3, 4), (3, 5), (4, 5), (5, 6), (5, 7), (6, 7)]
[4, 5, 3, 2, 1, 6, 7]
[(5, 2), (5, 1), (5, 7), (4, 3), (4, 1), (4, 6), (3, 2), (3, 1), (2, 1), (1, 6), (1, 7), (6, 7)]


In [ ]:
def isDAG(G: Graph) -> bool:
    """Returns true if the provided Graph is directed and acyclic.
    
    Returns:
        - bool True if G is a DAG, False otherwise"""
    topo_sort = topoSort2(G)
    return set(topo_sort)==set(G.nodes)

assert(isDAG(my_dag)==True)
assert(isDAG(my_tree)==False)
